<a href="https://colab.research.google.com/github/Gonzalo-Chable/practica2-FundamentosIA/blob/main/Proyecto_evacuacuion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Documentación de diseño y modelado del problema
1.1 Representación del terreno

El terreno se modela como una matriz 2D (numpy.ndarray de enteros), donde cada celda (fila, columna) representa una porción del mapa y contiene un valor de riesgo/costo:
Valor 	Significado 	Transitable 	Costo de entrar a la celda
0 	Zona bloqueada / inundación crítica (muro) 	❌ No 	— (infranqueable)
1 	Zona seca / segura 	✅ Sí 	1 (costo base)
3 	Inundación leve / riesgo bajo 	✅ Sí 	3
5 	Inundación moderada / riesgo medio 	✅ Sí 	5
10 	Inundación severa / riesgo alto 	✅ Sí 	10

Por convención, tanto 0 como -1 se tratan como bloqueados (un muro puede codificarse con cualquiera de los dos valores; en los mapas de este notebook usamos 0, pero el validador y los algoritmos reconocen ambos como intransitables).

El movimiento es ortogonal (4-conectividad: arriba, abajo, izquierda, derecha); no se permite movimiento diagonal, lo cual es importante tanto para el diseño de los mapas como para la heurística usada en A*.
1.2 La pregunta clave: ¿ruta más corta o ruta más conveniente?

    "Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?"

La respuesta está en qué función se minimiza:

    BFS minimiza el número de pasos (la longitud de la ruta), sin distinguir el nivel de riesgo de cada celda (solo evita las celdas totalmente bloqueadas). Esto lo hace rápido pero ciego al riesgo: puede escoger una ruta de 5 pasos que atraviesa inundación severa en vez de una ruta de 15 pasos completamente segura.

    Dijkstra y A* minimizan el costo acumulado g(n), definido como:

    g(n)=∑celda∈ruta hasta nriesgo(celda)

    Es decir, cada paso no "cuesta 1", sino que cuesta lo que indique el nivel de inundación de la celda a la que se entra. Una ruta de 5 pasos por zona de riesgo 10 tiene g = 50, mientras que una ruta de 15 pasos por zona segura (riesgo 1) tiene g = 15: el sistema la prefiere porque el costo acumulado — no el número de pasos — es lo que se minimiza.

Esto es precisamente lo que se explora a fondo en el Escenario 5.
1.3 A*: combinando g(n) con una heurística admisible

A* expande nodos ordenados por:

f(n)=g(n)+h(n)

donde h(n) es una estimación optimista (heurística) del costo restante desde n hasta el refugio. Usamos la distancia Manhattan, multiplicada por el costo mínimo posible de desplazamiento (que es 1, el costo de una celda segura):

h(n)=(|nx−rx|+|ny−ry|)×costo\_mínimo

con costo_mínimo = 1.

¿Por qué esto mantiene la admisibilidad? Una heurística es admisible si nunca sobreestima el costo real restante. Como el costo mínimo de entrar a cualquier celda transitable del mapa es 1 (zona segura), y la distancia Manhattan es el mínimo número de pasos ortogonales posible entre dos celdas (ignorando obstáculos), entonces:

h(n)=distancia\_Manhattan(n,refugio)×1≤costo real mínimo posible

Ningún camino real puede costar menos que "la cantidad mínima de pasos posibles, cada uno al costo mínimo posible". Por lo tanto h(n) siempre subestima o iguala el costo real, nunca lo sobreestima: la heurística es admisible, y A* garantiza encontrar la ruta de costo óptimo (igual que Dijkstra), pero típicamente expandiendo menos nodos, porque h(n) guía la búsqueda hacia el refugio en vez de explorar uniformemente en todas direcciones.

## 2. Implementación base, validador y algoritmos de búsqueda

A continuación se implementan:
- Utilidades de vecindad y reconstrucción de ruta.
- `validar_ruta`: validador programático e independiente de los algoritmos.
- `validar_entrada`: validador de integridad de datos de entrada (usado en el Escenario 6).
- Los tres algoritmos instrumentados: `bfs`, `dijkstra`, `a_estrella`.


In [ ]:
import heapq
import collections
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Valores de celda considerados bloqueados / intransitables
BLOQUEADO = {0, -1}

# Niveles de riesgo válidos (además de los bloqueos) usados en los mapas de este notebook
NIVELES_VALIDOS = {1, 3, 5, 10}


def vecinos_validos(mapa, pos):
    '''Genera las posiciones ortogonalmente adyacentes a `pos` que caen dentro del mapa.'''
    filas, cols = mapa.shape
    r, c = pos
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < filas and 0 <= nc < cols:
            yield (nr, nc)


def es_transitable(mapa, pos):
    '''Una celda es transitable si su valor no pertenece al conjunto BLOQUEADO.'''
    return mapa[pos] not in BLOQUEADO


def reconstruir_ruta(padres, inicio, fin):
    '''Reconstruye la ruta de `inicio` a `fin` siguiendo el diccionario de padres.'''
    ruta = [fin]
    actual = fin
    while actual != inicio:
        actual = padres[actual]
        ruta.append(actual)
    ruta.reverse()
    return ruta


def costo_real(mapa, ruta):
    '''
    Costo acumulado real de una ruta: suma del riesgo de cada celda VISITADA,
    sin contar la celda de inicio (no se 'paga' por empezar ahí, solo por entrar
    a cada celda siguiente).
    '''
    return sum(int(mapa[c]) for c in ruta[1:])
